# OSS MLOps workshop — Kubeflow + Feast (CPU-only)

Run cells **in order**. Edit **only** the configuration cell below (`WORKSHOP_NAMESPACE`).

Prerequisites: `oc` or `kubectl` authenticated; `feast` CLI installed in this environment; cluster has `ClusterTrainingRuntime` **torch-distributed**.

In [ ]:
from pathlib import Path

# --- attendee: set your namespace ---
WORKSHOP_NAMESPACE = "change-me-namespace"

# Kubeflow Pipelines UI + API (adjust if your cluster uses different names)
KFP_UI_NAMESPACE = "kubeflow"
KFP_ROUTE_NAME = "ml-pipeline-ui"
# Runs are created in the namespace where ml-pipeline runs (usually kubeflow for this install)
KFP_RUN_NAMESPACE = "kubeflow"

# Workshop root = parent of notebooks/
WORKSHOP_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
print("WORKSHOP_ROOT:", WORKSHOP_ROOT)
print("WORKSHOP_NAMESPACE:", WORKSHOP_NAMESPACE)

## 1) Feast — build parquet and `feast apply`

Feature definitions expect `data/transactions.parquet` (we generate it from the bundled CSV).

In [ ]:
import shutil
import subprocess
import sys

import pandas as pd

feast_repo = WORKSHOP_ROOT / "feast_repo"
csv_path = feast_repo / "data" / "transactions.csv"
parquet_path = feast_repo / "data" / "transactions.parquet"

df = pd.read_csv(csv_path)
df["amount"] = df["amount"].astype("float32")
df["is_fraud"] = df["is_fraud"].astype("float32")
df.to_parquet(parquet_path, index=False)
print("Wrote", parquet_path)

if not shutil.which("feast"):
    print("Install Feast CLI in this image, e.g. pip install 'feast[aws]' or your offline wheel", file=sys.stderr)
else:
    subprocess.run(["feast", "apply"], cwd=feast_repo, check=True)

## 2) Trainer v2 — ConfigMap + `TrainJob` (CPU, `torch-distributed`)

We still **reference** the cluster `torch-distributed` runtime; **CPU** image and training files are supplied via `trainer` + `podTemplateOverrides`.

In [ ]:
import subprocess

cli = "oc"  # use "kubectl" if you prefer

def apply_manifest(name: str) -> None:
    raw = (WORKSHOP_ROOT / "manifests" / name).read_text()
    raw = raw.replace("REPLACE_NAMESPACE", WORKSHOP_NAMESPACE)
    subprocess.run([cli, "apply", "-f", "-", "-n", WORKSHOP_NAMESPACE], input=raw.encode(), check=True)

apply_manifest("workshop-training-configmap.yaml")
apply_manifest("trainjob-fraud-workshop.yaml")
print("Applied ConfigMap + TrainJob")

In [ ]:
import subprocess

cli = "oc"
subprocess.run(
    [cli, "get", "trainjob", "-n", WORKSHOP_NAMESPACE],
    check=False,
)
# When READY/Succeeded, optionally copy the model out of the train pod:
# oc get pods -n $NS -l trainer.kubeflow.org/trainjob=fraud-workshop-train
# oc cp <pod>:/workspace/out/model.pt ./model.pt

## 3) Kubeflow Pipelines — compile, **visualize**, and open the UI

**Where to see the real KFP DAG (authoritative):** the **Kubeflow Pipelines web UI** — upload the compiled YAML (or use the optional submit cell below), start a run, then open the run’s **Graph** tab.

**In this notebook:** we (1) print a **clickable UI link** discovered via `oc get route`, (2) draw a **simple DAG figure** that matches this smoke pipeline (one task), and (3) optionally **submit a run** with `kfp` + `oc whoami -t` so you can jump straight to the run in the UI.

In [ ]:
import subprocess
import sys

pipe = WORKSHOP_ROOT / "pipeline" / "fraud_workshop_pipeline.py"
try:
    subprocess.run([sys.executable, str(pipe)], cwd=pipe.parent, check=True)
    print("Compiled:", pipe.with_suffix(".yaml"))
except subprocess.CalledProcessError as e:
    print("Compile failed — pip install 'kfp>=2.5,<3' in this environment.", file=sys.stderr)
    raise e

In [ ]:
import subprocess
from IPython.display import Markdown, display

cli = "oc"
try:
    host = subprocess.check_output(
        [cli, "get", "route", KFP_ROUTE_NAME, "-n", KFP_UI_NAMESPACE, "-o", "jsonpath={.spec.host}"],
        text=True,
    ).strip()
    ui = f"https://{host}"
    display(
        Markdown(
            f"### KFP web UI (upload YAML → **Create run** → **Graph**)\n\n"
            f"**[{ui}]({ui})**  _(same as `oc get route {KFP_ROUTE_NAME} -n {KFP_UI_NAMESPACE}`)_"
        )
    )
except subprocess.CalledProcessError:
    display(
        Markdown(
            "_Could not read Route; ask your admin for the `ml-pipeline-ui` URL "
            f"or run:_ `oc get route {KFP_ROUTE_NAME} -n {KFP_UI_NAMESPACE}`"
        )
    )

### In-notebook DAG (schematic)

This matches the **logical** shape of `fraud_workshop_pipeline` (one component). The **authoritative** task graph, retries, and status come from the **KFP UI** after you create a run.

In [ ]:
%pip install -q networkx matplotlib

import matplotlib.pyplot as plt
import networkx as nx

G = nx.DiGraph()
G.add_node("start", label="Pipeline start")
G.add_node("workshop_hello", label="workshop_hello\n(@dsl.component)")
G.add_node("end", label="Pipeline end")
G.add_edge("start", "workshop_hello")
G.add_edge("workshop_hello", "end")

pos = nx.spring_layout(G, seed=42)
labels = {n: G.nodes[n]["label"] for n in G.nodes()}
plt.figure(figsize=(7, 4))
nx.draw(
    G,
    pos,
    labels=labels,
    node_color="#e8f4fc",
    node_size=3800,
    font_size=9,
    arrows=True,
    arrowsize=16,
    edge_color="#333",
)
plt.title("fraud-workshop-pipeline (smoke — extend with more @dsl.component steps)")
plt.axis("off")
plt.tight_layout()
plt.show()

### Optional: submit a run from the notebook (then open **Graph** in the UI)

Uses `oc whoami -t` as the bearer token. Your user must be allowed to create runs in `KFP_RUN_NAMESPACE`.

In [ ]:
# Set SUBMIT_KFP_RUN = True after compile + UI link cells succeed
SUBMIT_KFP_RUN = False

import subprocess
from IPython.display import Markdown, display

if not SUBMIT_KFP_RUN:
    display(Markdown("_Set `SUBMIT_KFP_RUN = True` to create a run via the API._"))
else:
    try:
        from kfp.client import Client
    except ImportError:
        display(Markdown("_Install kfp:_ `pip install 'kfp>=2.5,<3'`"))
        raise

    host = subprocess.check_output(
        ["oc", "get", "route", KFP_ROUTE_NAME, "-n", KFP_UI_NAMESPACE, "-o", "jsonpath={.spec.host}"],
        text=True,
    ).strip()
    token = subprocess.check_output(["oc", "whoami", "-t"], text=True).strip()
    client = Client(host=f"https://{host}", existing_token=token, namespace=KFP_RUN_NAMESPACE)

    yaml_path = WORKSHOP_ROOT / "pipeline" / "fraud_workshop_pipeline.yaml"
    try:
        run = client.create_run_from_pipeline_package(
            pipeline_file=str(yaml_path),
            arguments={},
            experiment_name="fraud-workshop",
            namespace=KFP_RUN_NAMESPACE,
            job_name="fraud-workshop-notebook-run",
        )
    except TypeError:
        # Older kfp: no job_name
        run = client.create_run_from_pipeline_package(
            pipeline_file=str(yaml_path),
            arguments={},
            experiment_name="fraud-workshop",
            namespace=KFP_RUN_NAMESPACE,
        )

    run_id = getattr(run, "run_id", None) or getattr(run, "id", None)
    display(
        Markdown(
            f"**Run created.** Open the UI → **Runs** → find `fraud-workshop-notebook-run` "
            f"or run id `{run_id}` → **Graph** tab.\n\n"
            f"URL pattern (varies by KFP build): `https://{host}/#/runs/details/{{run_id}}`"
        )
    )
    print("run_id:", run_id)